In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.prebuilt import ToolNode, tools_condition

/home/rajab/Downloads/code/Langgraph-Notes/Chatbot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_60257/1773702155.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
load_dotenv()

True

In [4]:
# 1. Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [5]:
loader = PyPDFLoader("islr.pdf")
docs = loader.load()

In [6]:
len(docs)

441

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [8]:
len(chunks)

1308

In [10]:
# 3) Embed + index
embeddings = HuggingFaceEmbeddings(
    model="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1435.41it/s]


In [11]:
vector_store

In [12]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [13]:
@tool
def rag_tool(query):
    """
    Retrieve relevant information from the pdf document
    Use this tool when user asks conceptual / factual questions that might be answered from the stored documents.
    """
    result = retriever.invoke(query)
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    return {"query": query, "context": context, "metadata": metadata}


In [14]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    """LLM node that may answer or request a tool call"""

    messages: Annotated[list[BaseMessage], add_messages]


def chat_node(state: ChatState) -> ChatState:
    """LLM node that may answer or request a tool call"""
    messages = state["messages"]
    res = llm_with_tools.invoke(messages)
    return {"messages": [res]}


tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile()


In [17]:
res = chatbot.invoke({"messages": [HumanMessage("Hi, what is linear regression?")]})
print(res["messages"][-1].content)

Linear regression is a statistical method for modeling the relationship between a dependent variable and one or more independent variables. It is a linear approach for supervised learning, used for predicting quantitative values. The goal of linear regression is to create a linear equation that best predicts the value of the dependent variable based on the values of the independent variables. The equation is typically in the form of Y = β0 + β1X, where Y is the dependent variable, X is the independent variable, β0 is the intercept, and β1 is the slope. The coefficients β0 and β1 are determined by minimizing the sum of the squared errors between the observed and predicted values of Y. Linear regression is a widely used and important statistical technique, and it serves as a foundation for more complex statistical learning methods.
